# Auditing physical evidence in laboratory automation logs

**Research question:** To what extent can software-level laboratory actions be traced to independent evidence of their physical execution?

This notebook is a concise view of results computed by the normal Python modules in `src/lab_log_audit/`. The authoritative non-notebook entry point is `scripts/reproduce.py`. Headline numbers below are read from `results/metrics.json` and `results/window_sensitivity.csv`.

## 1. Dataset and provenance

The audit uses the Flex-Cat v1 archive (Zenodo record 18930287) and two modalities from Batch Distillation 1.1.2 (Zenodo record 21535243). Raw archives are not committed. `data/manifest.json` pins names, sizes, hashes, releases, retrieval date, creators, and licences.

Code in this repository is MIT-licensed. Flex-Cat and Batch Distillation are third-party CC BY 4.0 datasets; the MIT licence does not relicense them. Flex-Cat CC BY 4.0 is taken from the Zenodo API `metadata.license.id` for record 18930287 (checked 2026-08-30).

## 2. Definitions

- **Action:** a software-level `type="operation"` event, or a labelled recovery in metadata.
- **Controller/readback:** a value whose controller or device origin is established. The source does not establish that semantic class for `actualVolume`.
- **Independent physical evidence:** an observation from a distinct physical channel sufficient to support an effect claim.
- **RecoveryLabel / RecoveryEvidence / RecoveryOutcome:** kept distinct. A metadata recovery label is not evidence, and evidence in the operation log is not a proven outcome.
- **Window match:** any operation-log row whose parseable time of day lies inside the declared inclusive window. This is an observability/activity proxy. It is not evidence that the labelled recovery action itself was observed. A silent window is not evidence that no intervention occurred.

In [ ]:
import csv
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
metrics = json.loads((ROOT / "results" / "metrics.json").read_text(encoding="utf-8"))
with (ROOT / "results" / "window_sensitivity.csv").open(encoding="utf-8", newline="") as handle:
    sensitivity = list(csv.DictReader(handle))
inclusion = metrics["batch_distillation"]["inclusion"]
original = metrics["batch_distillation"]["original_window"]
actions = metrics["chemspeed"]["actions"]
endpoints = metrics["chemspeed"]["transfer_endpoints"]
print(metrics["research_question"])
print(metrics["batch_distillation"]["coverage_definition"])

## 3. Action-level audit

Identity is `(application_epoch, operationid)` for Chemspeed events with `type="operation"` only. Pairing is clean when that identity has exactly one start and one end.

In [ ]:
clean = actions["clean"]
print(f"Operations parsed: {actions['total']}")
print(f"Clean pairs: {clean['numerator']} / {clean['denominator']}")
print("Pairing status counts:", actions["pairing_status_counts"])

## 4. `actualVolume`

The comparison is exact decimal equality between requested `volume` and reported `actualVolume` on transfer endpoints that contain both fields. Equality does not establish an independent physical measurement, and it does not establish that the field is false.

In [ ]:
equal = endpoints["reported_equals_requested"]
print(
    f"Endpoints with reported actualVolume: {endpoints['with_reported_actual_volume']}"
)
print(f"Incomplete endpoints skipped: {endpoints['skipped_incomplete_endpoints']}")
print(f"Reported equals requested: {equal['numerator']} / {equal['denominator']}")
print("Evidence class:", endpoints["evidence_class"])
print(
    "Independent physical measurement established:",
    endpoints["independent_physical_measurement_established"],
)

## 5. Temporal evidence coverage

Two labelled recoveries are excluded because the pinned release has no operation log for that experiment. The coverage numerator counts included recoveries that have at least one parseable operation-log timestamp in the original inclusive window. That count is not a count of observed operator recoveries.

In [ ]:
print(f"Metadata labelled recoveries: {inclusion['metadata_labelled_recoveries']}")
print(f"Excluded without operation log: {inclusion['excluded_without_operation_log']}")
print("Excluded records:", inclusion["excluded_source_records"])
print(f"Included with operation log: {inclusion['included_with_operation_log']}")
print(
    "Unparseable or empty event timestamps:",
    inclusion["unparseable_event_timestamps"],
)
print(
    f"Original window [-{original['pre_window_seconds']} s, +{original['post_window_seconds']} s]: "
    f"{original['matched_actions']} / {original['total_actions']} = {original['coverage']:.2%}"
)

## 6. Sensitivity analysis

Wider windows can include unrelated UI or process activity. The table is a robustness check, not causal attribution, and still does not identify recovery actions.

In [ ]:
print("pre_s\tpost_s\tmatched\ttotal\tcoverage")
for row in sensitivity:
    coverage = float(row["coverage"])
    print(
        f"{row['pre_window_seconds']}\t{row['post_window_seconds']}\t"
        f"{row['matched_actions']}\t{row['total_actions']}\t{coverage:.2%}"
    )

## 7. Derived records

No separate manual-classification table was recovered. `data/derived/observations.csv` contains Chemspeed endpoint comparisons generated automatically (`review_method=automated_exact_decimal_comparison`). `results/recovery_windows.csv` exposes each included recovery match with row references. A `matched` row means log activity in the window, not an observed recovery action.

## 8. Limitations

This is a two-dataset audit, not a universal estimate for laboratory automation. Time-of-day matching does not infer midnight rollover. A matched row is not evidence of the intended physical effect. An unmatched window does not prove that no recovery occurred.